# Silver – Sales (bronze -> silver reference implementation)

Streams `car_workshop.fact.fact_sales_transactions` / `fact_sales_items` (bronze)
into validated, deduplicated silver tables. This notebook is the **pattern** –
the remaining silver tables (see `create_silver.sql` footer) are exercises.

**The pattern, per table:**
1. `readStream.table(bronze)` – incremental, checkpointed
2. `foreachBatch`:
   - dedup inside the batch (`dropDuplicates` on the business key)
   - data quality rules -> `_reject_reason` (first failing rule wins)
   - rejects -> quarantine table (append, with audit columns)
   - valid rows -> **insert-only `MERGE`** (idempotent: cross-batch duplicates
     and replayed batches are no-ops)
3. `trigger(availableNow=True)` – processes everything new, then stops

**Prerequisites:** `create_silver.sql` executed, bronze populated (autoloader).
Run order matters: transactions first, items second (items validate against
`silver.sales_transactions`).

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

CATALOG = 'car_workshop'
SILVER = f'{CATALOG}.silver'
CHECKPOINTS = f'/Volumes/{CATALOG}/silver/checkpoints'

# dims used for FK validation (small -> broadcast in joins)
dim_locations = spark.table(f'{CATALOG}.dim.dim_locations').select('location_id')
dim_employees = spark.table(f'{CATALOG}.dim.dim_employees').select('employee_id')
dim_products = spark.table(f'{CATALOG}.dim.dim_products').select(
    'product_id', 'category', 'purchase_price_net')

print(f'checkpoints: {CHECKPOINTS}')

## silver.sales_transactions

In [0]:
TRX_TARGET = f'{SILVER}.sales_transactions'
TRX_QUARANTINE = f'{SILVER}.sales_transactions_quarantine'

TRX_BRONZE_COLS = ['transaction_id', 'transaction_code', 'location_id', 'customer_id',
                   'employee_id', 'transaction_date', 'payment_method', 'receipt_number',
                   'year', 'month']


def process_sales_transactions(batch_df, batch_id):
    # 1. dedup inside the batch (bronze may contain exact duplicates)
    df = batch_df.dropDuplicates(['transaction_id'])

    # 2. data quality - first failing rule wins
    df = (df
          .join(F.broadcast(dim_locations.withColumn('_loc_ok', F.lit(True))), 'location_id', 'left')
          .join(F.broadcast(dim_employees.withColumn('_emp_ok', F.lit(True))), 'employee_id', 'left')
          .withColumn('_reject_reason',
                      F.when(F.col('transaction_id').isNull(), 'transaction_id is null')
                       .when(F.col('_loc_ok').isNull(), 'unknown location_id')
                       .when(F.col('_emp_ok').isNull(), 'unknown employee_id')))

    # 3. rejects -> quarantine (bronze shape + audit columns)
    (df.filter(F.col('_reject_reason').isNotNull())
       .select(*TRX_BRONZE_COLS, '_reject_reason')
       .withColumn('_quarantined_at', F.current_timestamp())
       .write.mode('append').saveAsTable(TRX_QUARANTINE))

    # 4. valid rows -> insert-only MERGE (idempotent across batches and re-runs)
    valid = (df.filter(F.col('_reject_reason').isNull())
               .select('transaction_id', 'transaction_code', 'location_id', 'customer_id',
                       'employee_id', 'transaction_date', 'payment_method', 'receipt_number')
               .withColumn('_processed_at', F.current_timestamp()))

    (DeltaTable.forName(spark, TRX_TARGET).alias('t')
     .merge(valid.alias('s'), 't.transaction_id = s.transaction_id')
     .whenNotMatchedInsertAll()
     .execute())


(spark.readStream
      .table(f'{CATALOG}.fact.fact_sales_transactions')
      .writeStream
      .foreachBatch(process_sales_transactions)
      .option('checkpointLocation', f'{CHECKPOINTS}/sales_transactions')
      .trigger(availableNow=True)
      .start()
      .awaitTermination())

print('silver.sales_transactions: done')

## silver.sales_items (enrichment + consistency checks)

In [0]:
ITEMS_TARGET = f'{SILVER}.sales_items'
ITEMS_QUARANTINE = f'{SILVER}.sales_items_quarantine'

ITEMS_BRONZE_COLS = ['sales_item_id', 'transaction_id', 'product_id', 'quantity',
                     'unit_price_net', 'discount_percent', 'value_net', 'vat_rate', 'value_gross']


def process_sales_items(batch_df, batch_id):
    df = batch_df.dropDuplicates(['sales_item_id'])

    # parents from silver (already validated) - re-read per batch, this table grows
    parents = (spark.table(TRX_TARGET)
               .select('transaction_id').withColumn('_trx_ok', F.lit(True)))

    df = (df
          .join(F.broadcast(dim_products.withColumn('_prod_ok', F.lit(True))), 'product_id', 'left')
          .join(parents, 'transaction_id', 'left')
          .withColumn('_expected_gross',
                      F.round(F.col('value_net') * (1 + F.col('vat_rate') / 100), 2))
          .withColumn('_reject_reason',
                      F.when(F.col('quantity') <= 0, 'non-positive quantity')
                       .when(F.col('_prod_ok').isNull(), 'unknown product_id')
                       .when(F.col('_trx_ok').isNull(), 'orphaned transaction_id')
                       .when(F.abs(F.col('value_gross') - F.col('_expected_gross')) > 0.02,
                             'gross/net mismatch')))

    (df.filter(F.col('_reject_reason').isNotNull())
       .select(*ITEMS_BRONZE_COLS, '_reject_reason')
       .withColumn('_quarantined_at', F.current_timestamp())
       .write.mode('append').saveAsTable(ITEMS_QUARANTINE))

    valid = (df.filter(F.col('_reject_reason').isNull())
               .withColumn('margin_net',
                           F.round(F.col('value_net') - F.col('purchase_price_net') * F.col('quantity'), 2))
               .withColumnRenamed('category', 'product_category')
               .select('sales_item_id', 'transaction_id', 'product_id', 'product_category',
                       'quantity', 'unit_price_net', 'discount_percent', 'value_net',
                       'vat_rate', 'value_gross', 'margin_net')
               .withColumn('_processed_at', F.current_timestamp()))

    (DeltaTable.forName(spark, ITEMS_TARGET).alias('t')
     .merge(valid.alias('s'), 't.sales_item_id = s.sales_item_id')
     .whenNotMatchedInsertAll()
     .execute())


(spark.readStream
      .table(f'{CATALOG}.fact.fact_sales_items')
      .writeStream
      .foreachBatch(process_sales_items)
      .option('checkpointLocation', f'{CHECKPOINTS}/sales_items')
      .trigger(availableNow=True)
      .start()
      .awaitTermination())

print('silver.sales_items: done')

## Validation

In [0]:
bronze_trx = spark.table(f'{CATALOG}.fact.fact_sales_transactions').count()
bronze_items = spark.table(f'{CATALOG}.fact.fact_sales_items').count()

for table, bronze_count in [('sales_transactions', bronze_trx), ('sales_items', bronze_items)]:
    silver_count = spark.table(f'{SILVER}.{table}').count()
    quarantined = spark.table(f'{SILVER}.{table}_quarantine').count()
    print(f'{table}: bronze {bronze_count:,} -> silver {silver_count:,} '
          f'(deduped {bronze_count - silver_count - quarantined:,}, quarantined {quarantined:,})')

In [0]:
# why were rows rejected?
display(spark.table(f'{SILVER}.sales_items_quarantine')
        .groupBy('_reject_reason').count().orderBy(F.desc('count')))

## Change Data Feed demo

Silver tables have CDF enabled – gold can consume inserts incrementally
instead of full scans.

In [0]:
last_version = spark.sql(f'DESCRIBE HISTORY {SILVER}.sales_transactions LIMIT 1').collect()[0]['version']
print(f'last table version: {last_version}')

display(spark.sql(f"""
    SELECT _change_type, count(*) AS rows
    FROM table_changes('{SILVER}.sales_transactions', {last_version})
    GROUP BY _change_type
"""))